<a href="https://colab.research.google.com/github/iqlore-collab/Weather-in-Cities-with-API-programmed-in-Visual-Studio-Code/blob/main/Weather_GUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import requests

# =========================
# CONFIG
# =========================
API_KEY = "PASTE YOURS API_KEY"


# =========================
# APP CLASS
# =========================
class WeatherApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Weather Desktop App")
        self.root.geometry("950x550")
        self.root.configure(bg="#1e1e2f")

        # ===== TITLE =====
        title = tk.Label(
            root,
            text="🌦 Weather Desktop App",
            font=("Helvetica", 20, "bold"),
            bg="#1e1e2f",
            fg="white"
        )
        title.pack(pady=10)

        # ===== INPUT =====
        input_frame = tk.Frame(root, bg="#1e1e2f")
        input_frame.pack(pady=10)

        self.city_entry = tk.Entry(input_frame, width=55, font=("Arial", 12))
        self.city_entry.insert(0, "Berlin, Hamburg, Munich, Cologne, Frankfurt")
        self.city_entry.grid(row=0, column=0, padx=10)

        self.btn = tk.Button(
            input_frame,
            text="Get Weather",
            command=self.get_weather,
            bg="#4CAF50",
            fg="white",
            font=("Arial", 12, "bold"),
            padx=10
        )
        self.btn.grid(row=0, column=1)

        # ===== TABLE =====
        columns = ("city", "temp", "feels_like", "humidity", "pressure", "wind", "description")

        self.tree = ttk.Treeview(root, columns=columns, show="headings", height=18)

        for col in columns:
            self.tree.heading(col, text=col.upper())
            self.tree.column(col, width=130)

        self.tree.pack(pady=20, fill="both", expand=True)

        # ===== STYLE =====
        style = ttk.Style()
        style.theme_use("default")

        style.configure("Treeview",
                        background="white",
                        foreground="black",
                        rowheight=25,
                        fieldbackground="white")

        style.configure("Treeview.Heading",
                        font=("Arial", 10, "bold"))

    # =========================
    # GET COORDINATES
    # =========================
    def get_coordinates(self, city):
        try:
            url = f"http://api.openweathermap.org/geo/1.0/direct?q={city},DE&limit=1&appid={API_KEY}"
            response = requests.get(url, timeout=10).json()

            if response:
                return response[0]["lat"], response[0]["lon"]

            return None, None

        except Exception:
            return None, None

    # =========================
    # GET WEATHER DATA
    # =========================
    def get_weather(self):
        cities = self.city_entry.get().split(",")

        # clear table
        self.tree.delete(*self.tree.get_children())

        for city in cities:
            city = city.strip()

            if not city:
                continue

            lat, lon = self.get_coordinates(city)

            if lat is None:
                messagebox.showerror("Error", f"City not found: {city}")
                continue

            try:
                url = (
                    f"https://api.openweathermap.org/data/2.5/forecast"
                    f"?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
                )

                data = requests.get(url, timeout=10).json()

                if "list" not in data:
                    messagebox.showerror("API Error", f"No data for {city}")
                    continue

                first = data["list"][0]

                self.tree.insert("", "end", values=(
                    city,
                    first["main"]["temp"],
                    first["main"]["feels_like"],
                    first["main"]["humidity"],
                    first["main"]["pressure"],
                    first["wind"]["speed"],
                    first["weather"][0]["description"]
                ))

            except Exception:
                messagebox.showerror("API Error", f"Failed for {city}")


# =========================
# RUN APP
# =========================
if __name__ == "__main__":
    root = tk.Tk()
    app = WeatherApp(root)
    root.mainloop()